In [1]:
import os
from datetime import datetime, timezone

import pandas as pd
import requests
from dotenv import load_dotenv

import domain

In [2]:
load_dotenv()

API_TOKEN = os.environ.get("CLASH_API_TOKEN")
if not API_TOKEN:
    raise RuntimeError("Falta CLASH_API_KEY no .env / env vars.")

HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}

def cr_get(url: str):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

In [3]:
def fetch_clan_members(clan_tag_no_hash: str) -> list[dict]:
    url = f"https://api.clashroyale.com/v1/clans/%23{clan_tag_no_hash}/members"
    data = cr_get(url)
    return data.get("items", data)

def fetch_player_battlelog(player_tag_no_hash: str) -> list[dict]:
    url = f"https://api.clashroyale.com/v1/players/%23{player_tag_no_hash}/battlelog"
    return cr_get(url)

CLAN_TAG = "QGVYLRVG"  # sem #
members = fetch_clan_members(CLAN_TAG)

# battlelogs em memória (dict: tag -> list[battles])
battles_by_player = {}
for m in members:
    tag_no_hash = m["tag"].replace("#", "")
    battles_by_player[tag_no_hash] = fetch_player_battlelog(tag_no_hash)

len(members), len(battles_by_player)


(50, 50)

In [4]:
from domain.models.player import Player
from domain.models.battle import Battle
from domain.filters.battle_filter import battle_weight

In [5]:
def normalize_tag(tag: str) -> str:
    return (tag or "").replace("#", "").strip()

def parse_battle_time_utc(b: dict) -> datetime:
    """
    Clash Royale battleTime típico: 'YYYYMMDDThhmmss.000Z'
    Ex: '20251220T212548.000Z'
    """
    s = (b.get("battleTime") or "").strip()
    if not s:
        return datetime.fromtimestamp(0, tz=timezone.utc)

    # Formato CR comum: 20251220T212548.000Z
    if len(s) >= 15 and s[8] == "T":
        iso = f"{s[0:4]}-{s[4:6]}-{s[6:8]}T{s[9:11]}:{s[11:13]}:{s[13:15]}"
        tail = s[15:]  # pode incluir ".000Z"
        if tail:
            iso += tail
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))

    # fallback
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


In [6]:
players: list[Player] = []

for m in members:
    tag_raw = m.get("tag", "")
    name = m.get("name", "")

    p = Player(tag=tag_raw, name=name)

    tag_no_hash = normalize_tag(tag_raw)

    # tenta buscar tanto por "#TAG" como por "TAG"
    raw_battles = (
        battles_by_player.get(tag_no_hash)
        or battles_by_player.get(tag_raw)
        or []
    )

    # cria Battles para o teu Player (o resto do pipeline fica no domain)
    p.battles = [
        Battle(
            timestamp=parse_battle_time_utc(b),
            battle_type=(b.get("type") or b.get("battleType") or ""),
            raw_json=b
        )
        for b in raw_battles
    ]

    players.append(p)

len(players), sum(len(p.battles) for p in players)


(50, 1668)

In [7]:
from domain.scoring.activity_score import compute_clan_baseline

snapshots = [p.activity_snapshot() for p in players]
baseline = compute_clan_baseline(snapshots)

rows = []
for p in players:
    snap = p.activity_snapshot()
    score = p.activity_score()

    eff_count = sum(1 for b in p.battles if battle_weight(b.raw_json) > 0.0)

    rows.append({
        "name": p.name,
        "tag": p.tag.replace("#", ""),
        "score": p.activity_score(clan_baseline=baseline),
        "days_since_last_effective": snap["days_since_last_effective"],
        "days_with_battles": snap["days_with_battles"],
        "raw_7d": snap.get("raw_7d"),
        "effective_7d": snap.get("effective_7d"),
        "weighted_7d": snap.get("weighted_7d"),
        "weighted_2d": snap.get("weighted_2d"),

        # sanity checks úteis
        "battles_total_fetched": len(p.battles),
        "effective_fetched": eff_count,
        "has_effective": eff_count > 0,
    })

df = pd.DataFrame(rows)

# ordenação: quem tem efetivas primeiro, depois score
df = df.sort_values(["has_effective", "score", "weighted_7d", "weighted_2d"], ascending=[False, False, False, False]).reset_index(drop=True)
df.insert(0, "rank", df.index + 1)

# opcional: não mostrar a coluna auxiliar
df.drop(columns=["has_effective"], inplace=True)

df.head(50)


TypeError: Player.activity_score() got an unexpected keyword argument 'clan_baseline'

In [8]:
weights = []
for p in players:
    for b in p.battles:
        weights.append(battle_weight(b.raw_json))

pd.Series(weights).value_counts().sort_index()


0.0     482
1.0    1186
Name: count, dtype: int64

In [10]:
import pandas as pd
from domain.scoring.activity_score import compute_clan_baseline, compute_activity_score

snapshots = [p.activity_snapshot() for p in players]
baseline = compute_clan_baseline(snapshots)

rows = []
for p in players:
    s = p.activity_snapshot()
    rows.append({
        "name": p.name,
        "tag": p.tag.replace("#", ""),
        "score": compute_activity_score(s),
        "weighted_7d": s["weighted_7d"],
        "weighted_2d": s["weighted_2d"],
        "days_since_last_effective": s["days_since_last_effective"],
        "battles_total_fetched": s["battles_total_fetched"],
        "days_with_battles": s["days_with_battles"],
        "effective_7d": s["effective_7d"],
        "raw_7d": s["raw_7d"],
    })

df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
df.insert(0, "rank", df.index + 1)
df.head(50)


,rank,name,tag,score,weighted_7d,weighted_2d,days_since_last_effective,battles_total_fetched,days_with_battles,effective_7d,raw_7d
0,1,1v0,LU0U2CU,1.000,30.0,30.0,0.523465,30,3,30,30
1,2,DUARTE LRVDS,2JY88QVJ,1.000,25.0,25.0,0.519483,30,3,25,30
2,3,Calisto,ULJJYPCPC,1.000,30.0,30.0,0.043407,30,3,30,30
3,4,Cross,9RY8PQVQ2,0.963,2.0,0.0,2.907504,30,3,2,30
4,5,RibeiroGamer,UL9YR09Q,0.943,30.0,0.0,3.331856,30,2,30,30
5,6,Ful8534,9090RV2JG,0.937,1.0,1.0,1.939287,30,2,1,30
6,7,Sr.Ferreira,LGGL0VG,0.936,28.0,18.0,0.056775,30,5,28,30
7,8,.sadud,2GCCC0V09,0.930,29.0,14.0,0.107574,30,4,29,30
8,9,fl1nt,2URGL09P9,0.914,21.0,9.0,0.992458,30,4,21,30
9,10,Rosario,V889Q8JC,0.904,30.0,0.0,2.747875,30,3,30,30


In [11]:
from domain.filters.battle_filter import should_ignore_battle
from domain.metrics.activity_metrics import battle_timestamp, battle_raw

CAP = 30

def normalize_tag(t: str) -> str:
    return t.replace("#", "").strip().upper()

def cap_last_n_battles(battles, n=30):
    # 1) remove battles proibidas (boatBattle etc.)
    battles = [b for b in battles if not should_ignore_battle(battle_raw(b))]

    # 2) ordena por timestamp (mais recente primeiro)
    battles = sorted(battles, key=battle_timestamp, reverse=True)

    # 3) cap nas últimas N
    return battles[:n]

# aplica ao dicionário inteiro
for tag, blist in list(battles_by_player.items()):
    battles_by_player[tag] = cap_last_n_battles(blist, CAP)

# sanity check
max_len = max((len(v) for v in battles_by_player.values()), default=0)
min_len = min((len(v) for v in battles_by_player.values()), default=0)
print("len range:", min_len, "->", max_len)


len range: 30 -> 30


In [12]:
import importlib
import domain.models.player as player_mod
importlib.reload(player_mod)
Player = player_mod.Player

players = []
for m in members:
    p = Player(m["tag"], m["name"])
    p.battles = battles_by_player.get(normalize_tag(m["tag"]), [])
    players.append(p)

len(players), len(players[0].battles) if players else None


(50, 30)

In [13]:
import importlib
import domain.filters.battle_filter as battle_filter
import domain.metrics.activity_metrics as activity_metrics
import domain.models.player as player_mod

importlib.reload(battle_filter)
importlib.reload(activity_metrics)
importlib.reload(player_mod)

# sanity: days_since_oldest deve ser <= ~15 (porque o battlelog só cobre ~15 dias)
bad = []
for p in players:
    s = p.activity_snapshot()
    if s["battles_total_fetched"] > 0 and s["days_since_oldest"] > 16:
        bad.append((p.name, s["battles_total_fetched"], s["days_since_oldest"], s.get("ignored_battles_removed")))

bad[:10], len(bad)


([('Bola Azul', 30, 17.70147930459491, 0),
  ('Chada', 30, 19.555125172662038, 0),
  ('TowerSniffer', 30, 20.084072360150465, 0),
  ('silviodias', 30, 24.761885022557873, 0),
  ('lord fenix', 30, 38.662174698090276, 0),
  ('Sannti', 30, 34.04455899207176, 0)],
 6)

In [76]:
s = players[0].activity_snapshot()
s.keys(), s.get("days_since_oldest")


(dict_keys(['battles_total_fetched', 'days_since_oldest', 'effective_total', 'effective_ratio_total', 'days_since_last_any', 'days_since_last_effective', 'days_with_battles', 'raw_7d', 'effective_7d', 'weighted_7d', 'weighted_2d', 'ignored_battles_removed']),
 1.1482742858680555)

In [14]:
# CELL 3 — construir players (assumindo que JÁ tens `members` e `battles_by_player` carregados no notebook)
# members: lista de dicts com pelo menos {"tag": "...", "name": "..."}
# battles_by_player: dict tag_normalizada -> lista de Battle (com .timestamp e .raw_json)

def normalize_tag(t: str) -> str:
    return t.replace("#", "").strip().upper()

players = []
for m in members:
    p = Player(tag=m["tag"], name=m["name"])
    p.battles = battles_by_player.get(normalize_tag(m["tag"]), [])
    players.append(p)

len(players), players[0].name if players else None


(50, '.sadud')

In [15]:
# CELL 4 — gerar tabela com snapshots + score
import pandas as pd

rows = []
for p in players:
    s = p.activity_snapshot()
    rows.append({
        "name": p.name,
        "tag": normalize_tag(p.tag),
        "score": p.activity_score(),

        "battles_total_fetched": s.get("battles_total_fetched"),
        "days_since_oldest": s.get("days_since_oldest"),
        "days_with_battles": s.get("days_with_battles"),

        "effective_ratio_total": s.get("effective_ratio_total"),
        "effective_total": s.get("effective_total"),

        "raw_7d": s.get("raw_7d"),
        "effective_7d": s.get("effective_7d"),
        "weighted_7d": s.get("weighted_7d"),
        "weighted_2d": s.get("weighted_2d"),

        "days_since_last_any": s.get("days_since_last_any"),
        "days_since_last_effective": s.get("days_since_last_effective"),
    })

df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
df.insert(0, "rank", df.index + 1)
df


,rank,name,tag,score,battles_total_fetched,days_since_oldest,days_with_battles,effective_ratio_total,effective_total,raw_7d,effective_7d,weighted_7d,weighted_2d,days_since_last_any,days_since_last_effective
0,1,1v0,LU0U2CU,1.000,30,1.894730,3,1.000000,30,30,30,30.0,30.0,0.524232,0.524232
1,2,Ruizao,UG9LPJUP,1.000,30,0.958361,2,1.000000,30,30,30,30.0,30.0,0.013442,0.013442
2,3,DUARTE LRVDS,2JY88QVJ,1.000,30,1.666266,3,0.833333,25,30,25,25.0,25.0,0.520248,0.520248
3,4,Calisto,ULJJYPCPC,1.000,30,1.841951,3,1.000000,30,30,30,30.0,30.0,0.044173,0.044173
4,5,_TASA_,VQCR2QCP,0.985,30,2.474589,3,1.000000,30,30,30,30.0,27.0,0.472575,0.472575
5,6,Rapanui,CJQ0J8JJV,0.970,30,2.978083,3,1.000000,30,30,30,30.0,14.0,0.922736,0.922736
6,7,LORD MATEUSX,C2CLQC8VC,0.967,30,2.905677,4,0.233333,7,30,7,7.0,5.0,0.024438,0.162181
7,8,EL Peterpan,CQ8C9P0GQ,0.963,30,3.182748,4,0.966667,29,30,29,29.0,15.0,0.401556,0.401556
8,9,Cross,9RY8PQVQ2,0.963,30,2.985816,3,0.066667,2,30,2,2.0,0.0,0.024439,2.908270
9,10,Dudas,28282PRLP,0.962,30,3.232169,4,1.000000,30,30,30,30.0,10.0,0.125097,0.125097


In [72]:
import domain.metrics.activity_metrics as am

# testa 3 batalhas do 1º player
sample = players[0].battles[:50]
print(players[0].name)
print(players[0].battles[::-5])
for i, b in enumerate(sample):
    print(i, type(b), am.battle_timestamp(b), "weight=", __import__("domain.filters.battle_filter").filters.battle_filter.battle_weight(am.battle_raw(b)))


Dudas
[{'type': 'boatBattle', 'battleTime': '20251114T030103.000Z', 'isLadderTournament': False, 'boatBattleSide': 'defender', 'boatBattleWon': True, 'newTowersDestroyed': 0, 'prevTowersDestroyed': 0, 'remainingTowers': 3, 'arena': {'id': 54000046, 'name': 'Legendary Arena', 'rawName': 'Arena_Clanboat'}, 'gameMode': {'id': 72000266, 'name': 'ClanWar_BoatBattle'}, 'deckSelection': 'collection', 'team': [{'tag': '#28282PRLP', 'name': 'Dudas', 'crowns': 3, 'kingTowerHitPoints': 0, 'princessTowersHitPoints': None, 'clan': {'tag': '#QGVYLRVG', 'name': 'FW Academy', 'badgeId': 16000169}, 'cards': [{'name': 'P.E.K.K.A', 'id': 26000004, 'level': 9, 'starLevel': 2, 'maxLevel': 11, 'maxEvolutionLevel': 1, 'rarity': 'epic', 'elixirCost': 7, 'iconUrls': {'medium': 'https://api-assets.clashroyale.com/cards/300/MlArURKhn_zWAZY-Xj1qIRKLVKquarG25BXDjUQajNs.png', 'evolutionMedium': 'https://api-assets.clashroyale.com/cardevolutions/300/MlArURKhn_zWAZY-Xj1qIRKLVKquarG25BXDjUQajNs.png'}}, {'name': 'Gobli

In [67]:
from domain.filters.battle_filter import should_ignore_battle

def battle_raw(b):
    return b.raw_json if hasattr(b, "raw_json") else b

count_boat = 0
for p in players:
    for b in p.battles:
        if should_ignore_battle(battle_raw(b)):
            count_boat += 1

print("boatBattle encontrados nas listas:", count_boat)


boatBattle encontrados nas listas: 0
